In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = "cuda"

In [ ]:
from pathlib import Path
from torchvision import datasets, transforms

DATA_DIR = Path("datasets")

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform)

print(len(train_dataset), len(test_dataset))

In [ ]:
def variance_schedule(T, s=0.008, max_beta=0.999):
    #alpha bare[t] -> alpha * alpha * alpha.... for t times
    t = torch.linspace(0, T, T + 1)
    f = torch.cos((t / T + s) / (1 + s) * torch.pi / 2) ** 2
    alpha_bars = f / f[0]
    betas = (1 - (f[1:] / f[:-1])).clamp(max=max_beta) #parrarel calculation from t = 0 to t=t
    betas = torch.cat([torch.zeros(1), betas])
    alphas = 1 - betas
    return alphas, betas, alpha_bars

T = 4000
alphas, betas, alpha_bars = variance_schedule(T)

In [ ]:
from collections import namedtuple

def forward_diffusion(x0, t):
    #xt = original image + t * noise step
    eps = torch.randn_like(x0)
    xt = alpha_bars[t].sqrt() * x0 + (1 - alpha_bars[t]).sqrt() * eps
    return xt, eps

class DiffusionSample(namedtuple("DiffusionSampleBase", ["xt", "t"])):
    def to(self, device):
        return DiffusionSample(self.xt.to(device), self.t.to(device))
class DiffusionDataset:
    def __init__(self, dataset):
        self.dataset = dataset
    
    def __getitem__(self, i):
        x0, _ = self.dataset[i]
        x0 = (x0 * 2) - 1 #scale from -1 to 1
        t = torch.randint(1, T+1, size=[1])
    
    def __len__(self):
        return len(self.dataset)

train_set = DiffusionDataset(train_dataset)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, pin_memory=True)

In [ ]:
def generate_ddim(model, batch_size=32, num_steps=50, eta=0.85):
    model.eval()
    with torch.no_grad():
        xt = torch.randn([batch_size, 1, 28, 28], device=device)
        times = torch.linspace(T - 1, 0, steps=num_steps + 1).long().tolist()
        for t, t_prev in zip(times[:-1], times[1:]):
            t_batch = torch.full((batch_size, 1), t, device=device)
            sample = DiffusionSample(xt, t_batch)
            eps_pred = model(sample)
            x0 = ((xt - (1 - alpha_bars)))